# AMANAH — Model Training & Validation Pipeline

End-to-end pipeline for source verification, dataset construction, training, calibration, held-out evaluation, deployment packaging, and optional Hugging Face upload.

The canonical Arabic source and trusted references remain immutable. Synthetic changes are created only as candidate samples. Metrics are reported only after the frozen test split is evaluated.


In [ ]:
# 1) Project setup
from pathlib import Path
from google.colab import userdata
import subprocess, os, json, shutil, platform, torch

repo_url = userdata.get('REPOSITORY_URL')
if not repo_url:
    raise RuntimeError('Add a Colab secret named REPOSITORY_URL with the Git repository clone URL, then run all cells again.')

project_dir = Path('/content/amanah')
if project_dir.exists(): shutil.rmtree(project_dir)
subprocess.run(['git','clone','--depth','1',repo_url,str(project_dir)], check=True)
os.chdir(project_dir)
subprocess.run(['python','-m','pip','install','-q','-r','requirements.txt'], check=True)
subprocess.run(['python','-m','pytest','-q'], check=True)

print('Python:', platform.python_version())
print('Torch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
assert torch.cuda.is_available(), 'GPU runtime required: Runtime > Change runtime type > T4 GPU.'
print('GPU:', torch.cuda.get_device_name(0))


In [ ]:
# 2) Verify external source contracts and fetch the reference bundle
subprocess.run(['python','scripts/source_contract_smoke.py','--timeout','90'], check=True)
Path('data/private').mkdir(parents=True,exist_ok=True)
Path('artifacts/amanah_sd').mkdir(parents=True,exist_ok=True)
Path('checkpoints').mkdir(parents=True,exist_ok=True)
subprocess.run(['python','scripts/fetch_verified_sources.py','--output','data/private/reference_bundle.json','--runtime-store','data/reference_store.json'], check=True)
bundle=json.load(open('data/private/reference_bundle.json',encoding='utf-8'))
assert len(bundle['ayahs']) == 6236
assert all(len(row['references']) == 3 for row in bundle['ayahs'])
print('Reference bundle verified:', len(bundle['ayahs']), 'ayahs')


In [ ]:
# 3) Build AMANAH-SD and run the dataset QA gate
subprocess.run(['python','-m','scripts.prepare_amanah_sd','data/private/reference_bundle.json','artifacts/amanah_sd','--seed','42'], check=True)
subprocess.run(['python','scripts/qa_dataset.py','--reference-bundle','data/private/reference_bundle.json','--split-dir','artifacts/amanah_sd','--output','artifacts/amanah_sd/qa_report.json'], check=True)
qa=json.load(open('artifacts/amanah_sd/qa_report.json',encoding='utf-8'))
assert qa['status'] == 'ok'
assert not qa['split_ayah_leakage']
print(json.dumps(qa,ensure_ascii=False,indent=2))


In [ ]:
# 4) Fine-tune the custom classifier
subprocess.run([
    'python','-m','training.train',
    '--train','artifacts/amanah_sd/train.jsonl',
    '--validation','artifacts/amanah_sd/validation.jsonl',
    '--output-dir','checkpoints/amanah-drift-v0.1',
    '--epochs','5','--batch-size','4','--gradient-accumulation','4',
    '--learning-rate','2e-5','--max-length','256','--patience','2'
], check=True)


In [ ]:
# 5) Calibrate on validation only, then evaluate once on the frozen test split
subprocess.run(['python','-m','training.calibrate_thresholds','--checkpoint','checkpoints/amanah-drift-v0.1','--validation','artifacts/amanah_sd/validation.jsonl'], check=True)
subprocess.run(['python','-m','training.evaluate','--checkpoint','checkpoints/amanah-drift-v0.1','--test','artifacts/amanah_sd/test.jsonl','--output','artifacts/evaluation'], check=True)
metrics=json.load(open('artifacts/evaluation/metrics.json',encoding='utf-8'))
assert metrics.get('status') == 'ok'
for key in ['macro_f1','critical_drift_recall','false_safe_rate','severity_macro_f1']:
    assert 0.0 <= metrics[key] <= 1.0
print(json.dumps(metrics,indent=2))


In [ ]:
# 6) Package the measured model for Hugging Face custom inference
subprocess.run(['python','scripts/package_hf_model.py','--checkpoint','checkpoints/amanah-drift-v0.1','--reference-store','data/reference_store.json','--output','artifacts/hf_model_repo'], check=True)
required=['handler.py','model.pt','amanah_config.json','thresholds.json','reference_store.json','requirements.txt']
missing=[name for name in required if not (Path('artifacts/hf_model_repo')/name).exists()]
assert not missing, f'Missing deployment files: {missing}'
print('Deployment package verified.')


In [ ]:
# 7) Local handler smoke test
import sys
sys.path.insert(0,'artifacts/hf_model_repo')
from handler import EndpointHandler
handler=EndpointHandler('artifacts/hf_model_repo')
store=json.load(open('artifacts/hf_model_repo/reference_store.json',encoding='utf-8'))
row=store['ayahs'][0]
response=handler({'inputs':{'source_type':'quran','source_ar':row['source_ar'],'candidate_en':row['references_en'][0],'ayah_id':row['ayah_id']}})
assert response['decision'] in {'PASS','REVIEW','CRITICAL','ABSTAIN'}
assert response['reference_status'] == 'verified'
print(json.dumps(response,ensure_ascii=False,indent=2))


In [ ]:
# 8) Persist final artifacts to Google Drive
from google.colab import drive
drive.mount('/content/drive')
drive_dir=Path('/content/drive/MyDrive/AMANAH_Artifacts')
drive_dir.mkdir(parents=True,exist_ok=True)
shutil.make_archive('/content/AMANAH_HF_DEPLOYMENT','zip','artifacts/hf_model_repo')
report={
  'dataset_summary':json.load(open('artifacts/amanah_sd/dataset_summary.json',encoding='utf-8')),
  'dataset_qa':json.load(open('artifacts/amanah_sd/qa_report.json',encoding='utf-8')),
  'evaluation':metrics,
  'hf_model_repo_url':None
}
Path('/content/AMANAH_FINAL_REPORT.json').write_text(json.dumps(report,ensure_ascii=False,indent=2),encoding='utf-8')
shutil.copy2('/content/AMANAH_FINAL_REPORT.json',drive_dir/'AMANAH_FINAL_REPORT.json')
shutil.copy2('/content/AMANAH_HF_DEPLOYMENT.zip',drive_dir/'AMANAH_HF_DEPLOYMENT.zip')
print('Artifacts saved to', drive_dir)


In [ ]:
# 9) Optional private Hugging Face model upload
from huggingface_hub import HfApi, create_repo
try:
    hf_token = userdata.get('HF_TOKEN')
except Exception:
    hf_token = None
hf_repo_url=None
if hf_token:
    api=HfApi(token=hf_token)
    identity=api.whoami()
    account=identity.get('name') or identity.get('fullname')
    repo_id=f'{account}/amanah-semantic-integrity-v0-1'
    create_repo(repo_id=repo_id,repo_type='model',private=True,exist_ok=True,token=hf_token)
    api.upload_folder(repo_id=repo_id,repo_type='model',folder_path='artifacts/hf_model_repo',token=hf_token,commit_message='Publish measured v0.1 checkpoint')
    hf_repo_url=f'https://huggingface.co/{repo_id}'
    report['hf_model_repo_url']=hf_repo_url
    Path('/content/AMANAH_FINAL_REPORT.json').write_text(json.dumps(report,ensure_ascii=False,indent=2),encoding='utf-8')
    shutil.copy2('/content/AMANAH_FINAL_REPORT.json',drive_dir/'AMANAH_FINAL_REPORT.json')
    print('Private model repository:', hf_repo_url)
else:
    print('HF_TOKEN not configured; model package remains ready for upload.')


In [ ]:
# 10) Final verification summary
print(json.dumps(report,ensure_ascii=False,indent=2))
print('Use only the measured values in the evaluation section for the presentation.')
